# Prueba individual · Sesión 03
### Career Tech Hub · Un caso de datos de principio a fin

**Nombre:** *Fabiola Altamirano Alfaro*

Trabajas con los mismos datos de la sesión. Lee antes el PDF de la prueba: ahí están las instrucciones y la rúbrica.

- Haz **al menos dos** de las partes A (pandas), B (SQL) y C (Excel o Power BI).
- La **parte D es obligatoria**.
- Este cuaderno tiene que estar en la misma carpeta que el cuaderno de la sesión, junto a la carpeta `datos`.

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pltly.express as px
rutaexcel = "/Users/fabiolaaltamirano/Sesion 3/Data/electricidad_para_panel.xlsx"
rutabbdd = "/Users/fabiolaaltamirano/Sesion 3/Data/electricidad.db"

# Las hojas del Excel (para la parte A)
dias = pd.read_excel(rutaexcel, sheet_name="dias")
generacion = pd.read_excel(rutaexcel, sheet_name="generacion")



# La base de datos (para la parte B)
conexion = sqlite3.connect(rutabbdd)

def sql(consulta):
    # Lanza una consulta y devuelve el resultado como tabla de pandas.
    return pd.read_sql(consulta, conexion)

print("Hoja dias:", dias.shape[0], "filas ·", list(dias.columns))
sql("SELECT type AS tipo, name AS nombre FROM sqlite_master WHERE type IN ('table', 'view') ORDER BY type, name")

Hoja dias: 1415 filas · ['fecha', 'dia_semana', 'mes', 'festivo', 'descanso', 't_med', 'demanda_gwh', 'año']


,tipo,nombre
0,table,ciudades
1,table,demanda
2,table,festivos
3,table,generacion
4,table,predicciones
5,table,tiempo
6,view,tabla_dias
7,view,temperatura_nacional


Columnas de las tablas de la base de datos:

| Tabla | Columnas |
|---|---|
| `demanda` | `fecha`, `demanda_gwh` |
| `generacion` | `fecha`, `tecnologia`, `tipo`, `gwh` |
| `tiempo` | `fecha`, `ciudad`, `t_max`, `t_min`, `t_med`, `lluvia_mm`, `viento_kmh`, `sol_mj` |
| `ciudades` | `ciudad`, `latitud`, `longitud`, `habitantes` |
| `festivos` | `fecha`, `festivo` |
| `temperatura_nacional` (vista) | `fecha`, `t_med` |
| `tabla_dias` (vista) | `fecha`, `demanda_gwh`, `t_med`, `festivo`, `demanda_ayer`, `demanda_hace_7` |

# Parte A · pandas

Usa la tabla `dias`.

**A1.** ¿Cuántos días superaron los 800 GWh de demanda? ¿Cuántos fueron en cada año?

In [ ]:

# Filtrar los días que superan los 800 GWh
dias_sup = dias[dias["demanda_gwh"] > 800]

# 3. Total de días
print(f"Días con demanda > 800 GWh: {len(dias_sup)}")

# 4. Días por año 
por_anio = (
    dias_sup
    .groupby("año")
    .size()
    .rename("dias_sup_800")
    .reset_index()
)
print(por_anio)

Días con demanda > 800 GWh: 115
    año  dias_sup_800
0  2023            16
1  2024            10
2  2025            25
3  2026            64


**A2.** Haz una tabla con la demanda media de cada año, separando los días laborables de los días de descanso (columna `descanso`). ¿Qué porcentaje menos se gasta en un día de descanso, cada año?

In [ ]:

# Demanda media por año, separando laborables y descanso
tabla = (
    dias.pivot_table(index="año", columns="descanso",
                   values="demanda_gwh", aggfunc="mean")
      .rename(columns={False: "laborable", True: "descanso"})
      .round(2)
)

# 3. Porcentaje menos de demanda en un día de descanso
tabla["pct_menos_descanso"] = (
    (tabla["laborable"] - tabla["descanso"]) / tabla["laborable"] * 100
).round(2)

print(tabla)

descanso  laborable  descanso  pct_menos_descanso
año                                              
2023         707.72    605.50               14.44
2024         710.06    612.02               13.81
2025         732.49    633.23               13.55
2026         749.86    643.03               14.25


**A3.** ¿Cuál fue el mes con mayor demanda media de los cuatro años (di el mes y el año)? ¿Y el de menor?

In [ ]:

#  Demanda media de cada mes de cada año
mensual = (
    dias.groupby(["año", "mes"])["demanda_gwh"]
      .mean()
      .round(2)
      .rename("demanda_media")
      .reset_index()
)

# 3. Mes con mayor y menor demanda media
mes_max = mensual.loc[mensual["demanda_media"].idxmax()]
mes_min = mensual.loc[mensual["demanda_media"].idxmin()]

print(f"Mayor demanda media: mes {int(mes_max['mes'])} de {int(mes_max['año'])} "
      f"-> {mes_max['demanda_media']} GWh")
print(f"Menor demanda media: mes {int(mes_min['mes'])} de {int(mes_min['año'])} "
      f"-> {mes_min['demanda_media']} GWh")

Mayor demanda media: mes 7 de 2026 -> 814.83 GWh
Menor demanda media: mes 4 de 2023 -> 610.56 GWh


# Parte B · SQL

Usa la función `sql("...")` con la base de datos.

**B1.** Saca los cinco días más calurosos (vista `temperatura_nacional`) junto con su demanda. ¿Son también días de mucha demanda? Explica por qué en una o dos frases.

In [ ]:

#  Cargar la vista temperatura_nacional y la tabla demanda
con = sqlite3.connect(rutabbdd)
temp = pd.read_sql("SELECT fecha, t_med FROM temperatura_nacional", con)
dem = pd.read_sql("SELECT fecha, demanda_gwh FROM demanda", con)
con.close()

# 2. Unir ambas por fecha (inner: solo días presentes en las dos)
df = temp.merge(dem, on="fecha", how="inner")

# 3. Los cinco días más calurosos con su demanda
top5 = df.nlargest(5, "t_med").reset_index(drop=True)
print(top5)

# 4. Para comparar: demanda media y percentil 75 de todos los días
print("\nDemanda media:", round(df["demanda_gwh"].mean(), 2))
print("Percentil 75:", round(df["demanda_gwh"].quantile(0.75), 2))



        fecha  t_med  demanda_gwh
0  2025-08-16  31.81   712.442720
1  2025-08-17  31.79   698.786118
2  2023-08-23  31.61   805.114493
3  2025-08-11  31.49   770.154701
4  2025-06-29  31.43   707.452281

Demanda media: 693.18
Percentil 75: 742.73


**B2.** Para cada año, calcula la generación solar fotovoltaica en TWh y qué porcentaje supone de toda la generación del año.

In [ ]:
con = sqlite3.connect(rutabbdd)

consulta = """
SELECT strftime('%Y', fecha) AS anio,
       ROUND(SUM(CASE WHEN tecnologia = 'Solar fotovoltaica' THEN gwh END) / 1000.0, 2) AS solar_twh,
       ROUND(SUM(gwh) / 1000.0, 2)                                                      AS total_twh,
       ROUND(100.0 * SUM(CASE WHEN tecnologia = 'Solar fotovoltaica' THEN gwh END)
             / SUM(gwh), 2)                                                             AS pct_solar
FROM generacion
GROUP BY anio
ORDER BY anio;
"""

df_solar = pd.read_sql(consulta, con)
con.close()

print(df_solar)


   anio  solar_twh  total_twh  pct_solar
0  2023      37.45     263.61      14.21
1  2024      44.60     262.43      16.99
2  2025      50.29     272.41      18.46
3  2026      55.41     247.37      22.40


**B3.** ¿Cuántos días de lluvia (1 mm o más) ha tenido cada ciudad? Ordena de más a menos.

In [32]:
con = sqlite3.connect(rutabbdd)

consulta = """
SELECT ciudad,
       COUNT(*) AS dias_lluvia
FROM tiempo
WHERE lluvia_mm >= 1
GROUP BY ciudad
ORDER BY dias_lluvia DESC;
"""

df_lluvia = pd.read_sql(consulta, con)
con.close()

print(df_lluvia)


      ciudad  dias_lluvia
0     Bilbao          611
1  Barcelona          350
2     Madrid          305
3   Zaragoza          269
4    Sevilla          239
5   Valencia          218


**Reto (opcional).** Con `LAG`, saca los tres días en que la demanda más subió respecto al día anterior. ¿Qué pasó en el primero?

In [33]:
con = sqlite3.connect(rutabbdd)

consulta = """
SELECT fecha,
       demanda_gwh,
       LAG(demanda_gwh, 1) OVER (ORDER BY fecha)               AS demanda_ayer,
       demanda_gwh - LAG(demanda_gwh, 1) OVER (ORDER BY fecha) AS subida_gwh
FROM demanda
ORDER BY subida_gwh DESC
LIMIT 3;
"""

df_subidas = pd.read_sql(consulta, con)
con.close()

print(df_subidas)


        fecha  demanda_gwh  demanda_ayer  subida_gwh
0  2025-04-29   590.008854    403.297490  186.711364
1  2026-01-07   849.726680    681.283230  168.443450
2  2023-07-10   778.800770    628.912612  149.888158


# Parte C · Excel o Power BI

Esta parte no se hace en el cuaderno. Abre `datos/electricidad_para_panel.xlsx` en Excel o cárgalo en Power BI.

**C1.** Haz una tabla dinámica (en Power BI, una matriz) con la demanda media: los meses en filas y los años en columnas. Hay una celda que no se puede comparar con las demás. ¿Cuál es y por qué?

**C2.** Haz un gráfico de la generación de 2025 por tecnología (hoja `generacion`), ordenado de mayor a menor. Ponle un título que diga la conclusión, no solo el tema.

Entrega una captura de cada una. Puedes pegarlas en un documento o guardarlas como imagen.

# Parte D · Explica (obligatoria)

Contesta con tus palabras. Frases cortas.

**D1.** Elige una decisión que hayas tenido que tomar en esta prueba y explica por qué la tomaste (4 a 6 líneas).

*(tu respuesta)*

**D2.** Cuenta tu resultado más interesante a alguien que no sabe de datos (3 líneas).

*(tu respuesta)*

**D3.** Di un límite de tu análisis, o algo que comprobarías si tuvieras más tiempo.

*(tu respuesta)*

**D4.** Si has usado una IA, di para qué y cómo comprobaste que el resultado era correcto. Si no la has usado, dilo.

*(tu respuesta)*